# Normal-form games & Nash equilibrium — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def softmax(x):
    x=np.asarray(x,dtype=float); e=np.exp(x-x.max()); return e/e.sum()
def expected_payoff(A,p,q): return float(np.asarray(p) @ np.asarray(A) @ np.asarray(q))
def best_response_payoffs(A,q): return np.asarray(A) @ np.asarray(q)
print('setup ok')

## Nash equilibrium in a two-player matrix game
A normal-form game is a payoff table plus one strategic question: is anyone able to improve by changing alone? These examples compute best responses, mark pure equilibria, solve the mixed equilibrium of matching pennies, follow best-response dynamics, and visualize the equilibrium support.

In [ ]:
# 1) Pure Nash equilibria: mutual best responses in a payoff matrix
A=np.array([[3,0],[5,1]])
B=np.array([[3,5],[0,1]])
row_br=(A==A.max(axis=0,keepdims=True)); col_br=(B==B.max(axis=1,keepdims=True))
nash=row_br & col_br
plt.figure(figsize=(4,3)); plt.imshow(nash,cmap='Greens',vmin=0,vmax=1)
plt.xticks([0,1],['C','D']); plt.yticks([0,1],['C','D']); plt.title('Prisoner dilemma: only (D,D) is Nash')
for i in range(2):
    for j in range(2): plt.text(j,i,f'({A[i,j]},{B[i,j]})',ha='center',va='center')
assert nash.tolist()==[[False,False],[False,True]]

In [ ]:
# 2) A mixed strategy changes expected payoffs linearly
A=np.array([[2,0],[0,1]])
qs=np.linspace(0,1,101); row0=2*qs; row1=1-qs
plt.figure(figsize=(5,3)); plt.plot(qs,row0,label='row plays Top'); plt.plot(qs,row1,label='row plays Bottom'); plt.axvline(1/3,ls='--',c='k')
plt.xlabel('column probability of Left'); plt.ylabel('row payoff'); plt.title('indifference at q=1/3'); plt.legend()
assert abs(row0[33]-0.66)<0.02 and abs((2/3)-(1-1/3))<1e-12

In [ ]:
# 3) Matching pennies has no pure Nash equilibrium but has a mixed one
A=np.array([[1,-1],[-1,1]]); B=-A
pure=[]
for i in range(2):
    for j in range(2):
        if A[i,j]==A[:,j].max() and B[i,j]==B[i,:].max(): pure.append((i,j))
ps=np.linspace(0,1,101); payoff_if_H=2*ps-1; payoff_if_T=1-2*ps
plt.figure(figsize=(5,3)); plt.plot(ps,payoff_if_H,label='column Heads payoff to row H'); plt.plot(ps,payoff_if_T,label='row T'); plt.axvline(.5,ls='--',c='k')
plt.xlabel('column probability Heads'); plt.ylabel('row payoff'); plt.title('mixed Nash: both use 0.5')
assert pure==[] and abs(expected_payoff(A,[.5,.5],[.5,.5]))<1e-12

In [ ]:
# 4) Best-response dynamics can cycle instead of converge
A=np.array([[1,-1],[-1,1]]); B=-A
i,j=0,0; path=[]
for t in range(8):
    path.append((i,j)); i=int(np.argmax(A[:,j])); j=int(np.argmax(B[i,:]))
coords=np.array(path)
plt.figure(figsize=(4,4)); plt.plot(coords[:,1],coords[:,0],'-o'); plt.xlim(-.2,1.2); plt.ylim(1.2,-.2); plt.xticks([0,1],['H','T']); plt.yticks([0,1],['H','T']); plt.title('best responses cycle')
assert path[1]==path[3] and len(set(path[:4]))==3

In [ ]:
# 5) Support check: at equilibrium each used action earns the same payoff
A=np.array([[1,-1],[-1,1]]); p=np.array([.5,.5]); q=np.array([.5,.5])
rp=A@q; cp=(-A).T@p
plt.figure(figsize=(5,3)); plt.bar(['row H','row T','col H','col T'],list(rp)+list(cp)); plt.axhline(0,c='k')
plt.title('all supported actions are indifferent at mixed Nash')
assert np.allclose(rp,[0,0]) and np.allclose(cp,[0,0])